# TP3: Denoising und Equalizer
### Labor Versuch TP3: Rauschen reduzieren und Klang verändern

In diesem Versuch fügen Sie einer sauberen Sprachaufnahme kontrollierte Störungen hinzu, verringern sie wieder und messen das Ergebnis. Danach verändern Sie den Klang von Musik mit einem Equalizer.

## 1 Hinweise zur Durchführung

- **Begleitheft:** Zu diesem Notebook gehört das PDF [`TP3_Denoising_Equalizer.pdf`](https://github.com/Lamhour-Mohamed-Akram/audio-dsp-lab/blob/main/handouts/TP3_Denoising_Equalizer.pdf). Abschnitte und Aufgaben haben im Notebook und im PDF **dieselben Nummern**.
- **Pflichtaufgaben** bearbeiten alle. **Zusatzaufgaben** sind freiwillig.
- **Antworten:** Schreiben Sie Ihre Antworten in das gedruckte Begleitheft **oder** in eine eigene Kopie dieses Notebooks. In Google Colab geht das über **Datei > Kopie in Drive speichern** (englisch *File > Save a copy in Drive*). Schreibrechte für das Original brauchen Sie nicht.
- **Zeitbedarf (Schätzung):** Pflichtaufgaben 3.1 bis 3.4 ca. 60 bis 90 Minuten nach der Einrichtung.
- **Arbeitsweise:** Führen Sie die Zellen von oben nach unten mit **Shift + Enter** aus. Ändern Sie in den Aufgaben immer nur **ein oder zwei Parameter** und führen Sie die Zelle danach erneut aus.
- **Kopfhörer:** Tiefe Töne sind über Laptop-Lautsprecher oft kaum hörbar.

**Hinweise für Google Colab**
- Eventuell müssen Sie sich mit einem Google-Konto anmelden.
- Beim ersten Ausführen kann eine Warnung erscheinen, dass das Notebook nicht von Google stammt. Bestätigen Sie mit **"Trotzdem ausführen"** (englisch *Run anyway*).
- **Sitzung neu starten** (englisch *Restart session*): Alle Variablen sind weg, die Dateien bleiben erhalten.
- **Laufzeit trennen und löschen** oder eine **neue Laufzeit**: Auch die heruntergeladenen Dateien sind weg.
- In beiden Fällen: Führen Sie die Zellen in Abschnitt 1 erneut aus und danach alle Zellen bis zu der Stelle, an der Sie arbeiten. Am einfachsten geht das mit **Laufzeit > Alle ausführen** (englisch *Runtime > Run all*).
- Eine GPU wird nicht benötigt. Dieses Notebook braucht kein anderes Notebook.

### 1.1 Einrichtung

Die nächsten zwei Zellen bereiten alles vor. Sie müssen den Code hier noch nicht verstehen.

1. Die erste Zelle prüft, ob alle benötigten Python-Pakete vorhanden sind. Nur fehlende Pakete werden installiert.
2. Die zweite Zelle stellt die Audiodateien `speech.wav`, `music.wav` im Ordner `data/` bereit. Sie lädt eine Datei **nur dann** herunter, wenn sie fehlt oder beschädigt ist.

In [ ]:
# Einrichtung 1: fehlende Pakete installieren (in Google Colab ist meist schon alles da)
import importlib.util
import subprocess
import sys

pakete = {"numpy": "numpy", "scipy": "scipy", "matplotlib": "matplotlib",
          "soundfile": "soundfile", "librosa": "librosa", "IPython": "ipython"}

fehlend = [paket for modul, paket in pakete.items() if importlib.util.find_spec(modul) is None]
if fehlend:
    print("Installiere:", ", ".join(fehlend))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *fehlend])
    print("Fertig.")
else:
    print("Alle Pakete sind bereits installiert.")

In [ ]:
# Einrichtung 2: Audiodateien bereitstellen (Download nur, wenn eine Datei fehlt oder beschädigt ist)
import os
import urllib.request
import soundfile as sf

DATEN_URL = "https://raw.githubusercontent.com/Lamhour-Mohamed-Akram/audio-dsp-lab/afe9e3d4fc8266c132421123ecd92235bc046779/data/"
ERWARTET = {"speech.wav": (16000, 1), "music.wav": (44100, 2)}   # Datei: (Abtastrate, Kanäle)


def audiodatei_ok(pfad, abtastrate, kanaele):
    """Prüft, ob die Datei eine lesbare Audiodatei im erwarteten Format ist."""
    try:
        info = sf.info(pfad)
    except Exception:
        return False
    return info.samplerate == abtastrate and info.channels == kanaele and info.frames > 0


os.makedirs("data", exist_ok=True)
for name, (abtastrate, kanaele) in ERWARTET.items():
    pfad = os.path.join("data", name)
    if audiodatei_ok(pfad, abtastrate, kanaele):
        print(name, ": vorhanden")
        continue
    print(name, ": wird heruntergeladen ...")
    try:
        with urllib.request.urlopen(DATEN_URL + name, timeout=60) as antwort:
            inhalt = antwort.read()
    except Exception as fehler:
        raise RuntimeError("Download von " + name + " fehlgeschlagen. Bitte Internetverbindung prüfen "
                           "und diese Zelle erneut ausführen. Grund: " + str(fehler))
    with open(pfad + ".tmp", "wb") as datei:
        datei.write(inhalt)
    if not audiodatei_ok(pfad + ".tmp", abtastrate, kanaele):
        os.remove(pfad + ".tmp")
        raise RuntimeError(name + " wurde geladen, ist aber keine gültige Audiodatei im erwarteten Format.")
    os.replace(pfad + ".tmp", pfad)
    print(name, ": OK")

print("Alle Audiodateien sind bereit.")

### 1.2 Bibliotheken und Hilfsfunktionen

Diese Zelle lädt die Bibliotheken und stellt kleine Hilfsfunktionen bereit, die Sie aus TP2 kennen:
- `anhoeren(signal, sr, text, skala)`: Abspieler **ohne** automatische Lautstärke-Anpassung
- `gemeinsame_skala(...)`: ein gemeinsamer Faktor, der Übersteuerung bei der Wiedergabe verhindert
- `sekunden_zu_samples(sekunden, sr)`: Zeit in Samples umrechnen
- `spektrum(signal, sr, glaettung_hz)`: einseitiges Amplitudenspektrum in dB (Sinus mit Amplitude 1 = 0 dB)
- `tiefpass(...)` und `hochpass(...)`: Butterworth-Filter, vorwärts und rückwärts angewendet

- `sosfiltfilt` wendet den Filter **vorwärts und rückwärts** an. Dadurch wird das Signal im Inneren **nicht zeitlich verschoben** (Nullphase).
- Die Dämpfung wirkt dabei **zweimal**: Der Betrag des Frequenzgangs wird quadriert. Bei der Grenzfrequenz ist das Signal deshalb um etwa 6 dB gedämpft statt um 3 dB.
- Ganz am Anfang und am Ende des Signals können kleine **Randeffekte** auftreten.
- `axis=0` sorgt dafür, dass bei Stereo jeder Kanal einzeln gefiltert wird. Das Stereobild bleibt erhalten.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import librosa

from IPython.display import Audio, display

def anhoeren(signal, sr, text="", skala=1.0):
    """Zeigt einen Audio-Abspieler, ohne automatische Lautstärke-Anpassung."""
    signal = np.asarray(signal) * skala
    if np.max(np.abs(signal)) > 1:
        print("Achtung: Werte über 1 werden für die Wiedergabe abgeschnitten. Bitte kleinere Skala wählen.")
        signal = np.clip(signal, -1, 1)
    if signal.ndim == 2:        # Stereo: der Abspieler erwartet (Kanäle, Samples)
        signal = signal.T
    if text:
        print(text)
    display(Audio(signal, rate=sr, normalize=False))

def gemeinsame_skala(*signale):
    """Ein gemeinsamer Faktor für alle Signale. Er ist nur kleiner als 1, wenn sonst etwas übersteuern würde."""
    spitze = max(np.max(np.abs(s)) for s in signale)
    if spitze == 0:
        return 1.0
    return min(1.0, 0.99 / spitze)

def sekunden_zu_samples(sekunden, abtastrate):
    """Rechnet eine Zeit in Sekunden in eine Anzahl von Samples um."""
    anzahl = int(round(sekunden * abtastrate))    # round() rundet, int() macht eine ganze Zahl daraus
    return anzahl

In [ ]:
def spektrum(signal, sr, glaettung_hz=0):
    """Einseitiges Amplitudenspektrum in dB. Ein Sinus mit Amplitude 1 ergibt 0 dB."""
    if signal.ndim == 2:
        signal = signal.mean(axis=1)                 # Stereo zu Mono (nur für die Analyse)
    n = len(signal)
    if n < 2:
        raise ValueError("Das Signal ist zu kurz für eine FFT (mindestens 2 Samples).")
    betrag = np.abs(np.fft.rfft(signal)) / n
    betrag[1:] = 2 * betrag[1:]                      # positive Frequenzen: Anteil der negativen dazuzählen
    if n % 2 == 0:
        betrag[-1] = betrag[-1] / 2                  # die Nyquist-Frequenz gibt es nur einmal
    frequenzen = np.fft.rfftfreq(n, d=1 / sr)
    breite = int(glaettung_hz * n / sr)              # Glättung über glaettung_hz Hertz (0 = aus)
    if breite > 1:
        betrag = np.convolve(betrag, np.ones(breite) / breite, mode="same")
    return frequenzen, 20 * np.log10(betrag + 1e-12)

from scipy import signal as sig

def pruefe_grenzfrequenz(grenzfrequenz, sr):
    """Gültig sind Grenzfrequenzen größer als 0 und kleiner als die halbe Abtastrate."""
    if not 0 < grenzfrequenz < sr / 2:
        raise ValueError("Grenzfrequenz " + str(grenzfrequenz) + " Hz ist ungültig. Erlaubt: größer als 0 "
                         "und kleiner als " + str(sr / 2) + " Hz (halbe Abtastrate).")

def tiefpass(x, sr, grenzfrequenz, ordnung=4):
    """Butterworth-Tiefpass, vorwärts und rückwärts angewendet."""
    pruefe_grenzfrequenz(grenzfrequenz, sr)
    sos = sig.butter(ordnung, grenzfrequenz, btype="lowpass", fs=sr, output="sos")
    return sig.sosfiltfilt(sos, x, axis=0)

def hochpass(x, sr, grenzfrequenz, ordnung=4):
    """Butterworth-Hochpass, vorwärts und rückwärts angewendet."""
    pruefe_grenzfrequenz(grenzfrequenz, sr)
    sos = sig.butter(ordnung, grenzfrequenz, btype="highpass", fs=sr, output="sos")
    return sig.sosfiltfilt(sos, x, axis=0)

## 2 Lernziele

Nach diesem Versuch können Sie ...
- Störungen **reproduzierbar** zu einer sauberen Sprachaufnahme hinzufügen,
- ein **Brummen** mit einem Hochpass verringern und den Nachteil für die Sprache erklären,
- **breitbandiges Rauschen** mit der **spektralen Subtraktion** verringern,
- das **SNR** gegenüber einer sauberen Referenz berechnen,
- **Signalähnlichkeit**, **Verständlichkeit** und **empfundene Qualität** unterscheiden,
- Musik mit einem **Equalizer** verändern und **lauter** von **besser** unterscheiden.

## 3 Vorbereitungsaufgaben

- **V1:** Das SNR in dB ist 10 mal log10(Leistung des Signals / Leistung der Störung). log10 ist der Logarithmus zur Basis 10. Wie groß ist das SNR, wenn beide **gleich viel** Leistung haben? Und wenn das Signal **10-mal** so viel Leistung hat?
- **V2:** Das Stromnetz in Europa hat 50 Hz. Welche Frequenzen hat ein Brummen aus dem Grundton und seinen ersten zwei Vielfachen?
- **V3:** **Weißes Rauschen** enthält alle Frequenzen ungefähr gleich stark. Warum kann ein Tiefpass es **nicht vollständig** entfernen?
- **V4:** Eine verrauschte Aufnahme wird doppelt so laut gemacht. Was passiert mit dem Verhältnis von Sprache zu Rauschen?

**Ihre Antworten:**
- V1: 
- V2: 
- V3: 
- V4: 

## 4 Durchführung

### 4.1 Saubere Referenz und SNR

`speech.wav` ist unsere **saubere Referenz**. Wir fügen selbst Störungen hinzu. Weil wir das saubere Signal kennen, können wir den Fehler genau messen:

- Fehler = bearbeitetes Signal - sauberes Signal
- **SNR in dB** = 10 mal log10(Leistung des sauberen Signals / Leistung des Fehlers)

Ein größeres SNR bedeutet: Das Ergebnis ist dem sauberen Signal **ähnlicher**. **Jede** Abweichung zählt als Fehler, auch wenn ein Filter nützliche Teile der Sprache entfernt.

Drei Dinge muss man unterscheiden:
- **Signalähnlichkeit:** Wie nah ist das Ergebnis am sauberen Signal? Das misst das SNR.
- **Verständlichkeit:** Kann man die Wörter verstehen?
- **Empfundene Qualität:** Klingt es angenehm und natürlich?

Diese drei Eigenschaften hängen zusammen, sind aber **nicht dasselbe**. Verständlichkeit und Qualität beurteilen wir hier durch **Hören**.

*In der Praxis kennt man das saubere Signal meist nicht. Dieser Versuch ist deshalb eine Laborsituation.*

In [ ]:
def snr_db(sauber, test):
    """SNR in dB gegenüber der sauberen Referenz (beide Signale gleich lang)."""
    if len(sauber) != len(test):
        raise ValueError("Beide Signale müssen gleich lang sein.")
    fehler = test - sauber
    leistung_fehler = np.sum(fehler**2)
    if leistung_fehler == 0:
        return np.inf                      # identisch: kein Fehler
    return 10 * np.log10(np.sum(sauber**2) / leistung_fehler)

sprache, sr_sprache = sf.read("data/speech.wav")
zeit_sprache = np.arange(len(sprache)) / sr_sprache      # Zeitachse in Sekunden
anhoeren(sprache, sr_sprache, "Saubere Referenz:")

### 4.2 Brummen und Hochpass (Aufgabe 3.1, Pflicht)

Wir addieren ein **Brummen** aus 50 Hz, 100 Hz und 150 Hz. Diese Störung liegt nur bei **tiefen** Frequenzen.

In [ ]:
brummen = (0.03 * np.sin(2 * np.pi * 50 * zeit_sprache)
           + 0.02 * np.sin(2 * np.pi * 100 * zeit_sprache)
           + 0.01 * np.sin(2 * np.pi * 150 * zeit_sprache))

mit_brummen = sprache + brummen
print("SNR mit Brummen:", round(snr_db(sprache, mit_brummen), 1), "dB")

f_sauber, db_sauber = spektrum(sprache, sr_sprache, glaettung_hz=2)
f_brumm, db_brumm = spektrum(mit_brummen, sr_sprache, glaettung_hz=2)
plt.figure(figsize=(10, 3))
plt.plot(f_brumm, db_brumm, label="mit Brummen", linewidth=0.8)
plt.plot(f_sauber, db_sauber, label="sauber", linewidth=0.8)
plt.xlim(0, 1000)
plt.ylim(-110, -20)
plt.xlabel("Frequenz [Hz]")
plt.ylabel("Stärke [dB], geglättet über 2 Hz")
plt.title("Spektrum mit und ohne Brummen")
plt.legend()
plt.grid(True)
plt.show()

Ein **Hochpass** dämpft die tiefen Frequenzen und damit das Brummen. **Aber:** Die Stimme hat ebenfalls Anteile bei tiefen Frequenzen. Je höher die Grenzfrequenz, desto mehr nützliche Sprache wird mit entfernt. Wir benutzen einen steileren Filter (Ordnung 6).

In [ ]:
grenzfrequenz_brumm = 200    # Hz, Vorschläge: 60, 120, 200, 400, 800 (gültig: größer als 0, kleiner als 8000)

ohne_brummen = hochpass(mit_brummen, sr_sprache, grenzfrequenz_brumm, ordnung=6)

snr_vorher = snr_db(sprache, mit_brummen)
snr_nachher = snr_db(sprache, ohne_brummen)
print("SNR vorher: ", round(snr_vorher, 1), "dB")
print("SNR nachher:", round(snr_nachher, 1), "dB")
print("Änderung:   ", round(snr_nachher - snr_vorher, 1), "dB (positiv = gestiegen, negativ = gesunken)")

skala_brumm = gemeinsame_skala(sprache, mit_brummen, ohne_brummen)
anhoeren(sprache, sr_sprache, "Sauber:", skala_brumm)
anhoeren(mit_brummen, sr_sprache, "Mit Brummen:", skala_brumm)
anhoeren(ohne_brummen, sr_sprache, "Nach dem Hochpass:", skala_brumm)

> **Aufgabe 3.1:** Probieren Sie `grenzfrequenz_brumm` = 60, 120, 200, 400 und 800. Tragen Sie in **Tabelle A** ein: SNR nachher, ob das SNR gestiegen oder gesunken ist, ob das Brummen noch hörbar ist und wie sich die Stimme verändert. Welche Grenzfrequenz ist für Sie der beste Kompromiss? Begründen Sie mit Ihren Messwerten **und** Ihrem Höreindruck.

#### Zusatzaufgabe Z3.1: Kerbfilter (optional)

Ein **Kerbfilter** (englisch *notch*) dämpft nur einen sehr schmalen Frequenzbereich um eine Frequenz herum. Mit drei Kerbfiltern bei 50, 100 und 150 Hz bleibt der Rest der Stimme weitgehend erhalten. Der **Gütefaktor** `Q` bestimmt die Breite: größeres `Q` bedeutet schmalere Kerbe (sinnvoll etwa 5 bis 100).

In [ ]:
kerb_q = 30        # Vorschläge: 5, 30, 100

kerb = mit_brummen.copy()
for frequenz in [50, 100, 150]:
    b, a = sig.iirnotch(frequenz, Q=kerb_q, fs=sr_sprache)
    kerb = sig.filtfilt(b, a, kerb)

print("SNR nach den Kerbfiltern:", round(snr_db(sprache, kerb), 1), "dB")
anhoeren(kerb, sr_sprache, "Nach den Kerbfiltern:", skala_brumm)

> **Z3.1:** Vergleichen Sie `kerb_q` = 5, 30 und 100 mit dem besten Hochpass aus Aufgabe 3.1. Was beobachten Sie beim SNR und beim Klang?

### 4.3 Breitbandiges Rauschen (Aufgabe 3.2, Pflicht)

**Weißes Rauschen** (wie Rauschen im Radio zwischen zwei Sendern) enthält **alle Frequenzen**, also auch die Frequenzen der Sprache. Wir erzeugen es mit Zufallszahlen.

- Mit `seed=42` erzeugt der Zufallsgenerator **jedes Mal dasselbe** Rauschen. So sind Ergebnisse reproduzierbar und zwischen Gruppen vergleichbar.
- `ziel_snr_db` legt fest, wie stark das Rauschen ist.
- Zusätzlich wird automatisch eine **Rauschprobe** von 1 Sekunde erzeugt. Das ist so, als hätte das Mikrofon vor dem Sprechen kurz nur das Rauschen aufgenommen. Diese Probe braucht die spektrale Subtraktion in 4.4.

In [ ]:
ziel_snr_db = 10         # gewünschtes SNR in dB, Vorschläge: 0, 10, 20

rng = np.random.default_rng(seed=42)          # fester Startwert: reproduzierbares Rauschen
leistung_sprache = np.mean(sprache**2)
rausch_staerke = np.sqrt(leistung_sprache / 10**(ziel_snr_db / 10))

rauschen = rausch_staerke * rng.normal(0, 1, len(sprache))
rauschprobe = rausch_staerke * rng.normal(0, 1, sr_sprache)      # 1 Sekunde nur Rauschen

verrauscht = sprache + rauschen
print("SNR der verrauschten Sprache:", round(snr_db(sprache, verrauscht), 1), "dB")

skala_rauschen = gemeinsame_skala(sprache, verrauscht)
anhoeren(sprache, sr_sprache, "Sauber:", skala_rauschen)
anhoeren(verrauscht, sr_sprache, "Verrauscht:", skala_rauschen)

**Was macht ein Tiefpass mit breitbandigem Rauschen?** Ein Tiefpass dämpft nur Frequenzen **oberhalb** der Grenzfrequenz. Rauschen unterhalb der Grenzfrequenz, also im Bereich der Sprache, bleibt erhalten. Gleichzeitig werden hohe Anteile der Sprache gedämpft, zum Beispiel bei "s", "f" und "sch".

In [ ]:
grenzfrequenz_tp = 3000   # Hz, Vorschläge: 1000, 3000, 6000 (gültig: größer als 0, kleiner als 8000)

verrauscht_tp = tiefpass(verrauscht, sr_sprache, grenzfrequenz_tp)

print("SNR verrauscht:       ", round(snr_db(sprache, verrauscht), 1), "dB")
print("SNR nach dem Tiefpass:", round(snr_db(sprache, verrauscht_tp), 1), "dB")
anhoeren(verrauscht, sr_sprache, "Verrauscht:", skala_rauschen)
anhoeren(verrauscht_tp, sr_sprache, "Verrauscht und Tiefpass:", skala_rauschen)

> **Aufgabe 3.2:** Probieren Sie `grenzfrequenz_tp` = 1000, 3000 und 6000. Ist das SNR gestiegen oder gesunken? Ist das Rauschen noch hörbar? Wie verändert sich die Sprache? Notieren Sie Ihre Beobachtungen in **Tabelle B**.

### 4.4 Spektrale Subtraktion (Aufgabe 3.3, Pflicht)

Die Idee in vier Schritten. Sie müssen das Verfahren **nicht** selbst programmieren. Die fertige Funktion `spektrale_subtraktion` ist unten erklärt.
1. Das verrauschte Signal wird in viele kurze, überlappende Stücke (32 ms) zerlegt. Für jedes Stück wird ein Spektrum berechnet.
2. Aus der **Rauschprobe** wird berechnet, wie stark das Rauschen im Mittel bei jeder Frequenz ist.
3. Von jedem Spektrum wird diese Rauschstärke **abgezogen**. Ein **Boden** legt fest, wie viel mindestens übrig bleibt.
4. Aus den bereinigten Spektren wird wieder ein Zeitsignal.

Das funktioniert am besten bei **gleichmäßigem** Rauschen, dessen Klang sich nicht ändert. Es entfernt **nicht** jede Störung. Zu starkes Abziehen erzeugt **musikalisches Rauschen**, das wie Zwitschern oder Blubbern klingt.

Parameter:
- `staerke`: Faktor für das abgezogene Rauschspektrum (0 = nichts abziehen, 1.0 = mittlere Rauschstärke; sinnvoll 0 bis 5)
- `boden`: Anteil des ursprünglichen Spektrums, der mindestens bleibt (0.05 = 5 %; gültig 0 bis 1)

In [ ]:
def spektrale_subtraktion(verrauscht, rauschprobe, staerke=1.0, boden=0.05):
    """Einfache spektrale Subtraktion mit einer Probe, die nur Rauschen enthält (für 16000 Hz ausgelegt)."""
    if staerke < 0:
        raise ValueError("staerke muss 0 oder größer sein.")
    if not 0 <= boden <= 1:
        raise ValueError("boden muss zwischen 0 und 1 liegen.")
    n_fft, hop = 512, 128                                     # 512 Samples = 32 ms bei 16000 Hz
    if len(rauschprobe) < n_fft:
        raise ValueError("Die Rauschprobe ist zu kurz (mindestens 512 Samples).")
    # 1) Kurzzeit-Spektren des verrauschten Signals: Betrag und Phase
    stft = librosa.stft(verrauscht, n_fft=n_fft, hop_length=hop)
    betrag, phase = np.abs(stft), np.angle(stft)
    # 2) mittleres Rauschspektrum aus der Rauschprobe
    rausch_betrag = np.abs(librosa.stft(rauschprobe, n_fft=n_fft, hop_length=hop)).mean(axis=1, keepdims=True)
    # 3) Rauschen abziehen, aber nie unter den Boden fallen
    neuer_betrag = np.maximum(betrag - staerke * rausch_betrag, boden * betrag)
    # 4) zurück in den Zeitbereich (die Phase des verrauschten Signals wird behalten)
    return librosa.istft(neuer_betrag * np.exp(1j * phase), hop_length=hop, length=len(verrauscht))

In [ ]:
staerke = 1.0     # Vorschläge: 0.5, 1.0, 2.0, 4.0
boden = 0.05      # Zusatz: 0.01 oder 0.2

entrauscht = spektrale_subtraktion(verrauscht, rauschprobe, staerke, boden)

snr_vorher = snr_db(sprache, verrauscht)
snr_nachher = snr_db(sprache, entrauscht)
print("SNR verrauscht:", round(snr_vorher, 1), "dB")
print("SNR entrauscht:", round(snr_nachher, 1), "dB")
print("Änderung:      ", round(snr_nachher - snr_vorher, 1), "dB (positiv = gestiegen, negativ = gesunken)")

skala_ss = gemeinsame_skala(sprache, verrauscht, entrauscht)
anhoeren(sprache, sr_sprache, "Sauber:", skala_ss)
anhoeren(verrauscht, sr_sprache, "Verrauscht:", skala_ss)
anhoeren(entrauscht, sr_sprache, "Nach spektraler Subtraktion:", skala_ss)

In [ ]:
von_s, bis_s = 3.0, 5.5          # gezeigter Zeitbereich in Sekunden

fig, achsen = plt.subplots(3, 1, figsize=(10, 6), sharex=True, sharey=True)   # 3 Diagramme untereinander
achsen[0].plot(zeit_sprache, sprache, linewidth=0.5)
achsen[0].set_title("Sauber")
achsen[1].plot(zeit_sprache, verrauscht, linewidth=0.5, color="tab:red")
achsen[1].set_title("Verrauscht")
achsen[2].plot(zeit_sprache, entrauscht, linewidth=0.5, color="tab:green")
achsen[2].set_title("Nach spektraler Subtraktion")
achsen[2].set_xlim(von_s, bis_s)
achsen[2].set_xlabel("Zeit [s]")
plt.tight_layout()
plt.show()

f_s, db_s = spektrum(sprache, sr_sprache, glaettung_hz=20)
f_v, db_v = spektrum(verrauscht, sr_sprache, glaettung_hz=20)
f_e, db_e = spektrum(entrauscht, sr_sprache, glaettung_hz=20)
plt.figure(figsize=(10, 3))
plt.plot(f_v, db_v, label="verrauscht", color="tab:red", linewidth=0.8)
plt.plot(f_e, db_e, label="entrauscht", color="tab:green", linewidth=0.8)
plt.plot(f_s, db_s, label="sauber", color="tab:blue", linewidth=0.8)
plt.xlim(0, 8000)
plt.ylim(np.max(db_s) - 70, np.max(db_s) + 5)
plt.xlabel("Frequenz [Hz]")
plt.ylabel("Stärke [dB], geglättet")
plt.title("Spektren im Vergleich (gleiche Skala, nicht einzeln normiert)")
plt.legend()
plt.grid(True)
plt.show()

> **Aufgabe 3.3:** Probieren Sie `staerke` = 0.5, 1.0, 2.0 und 4.0 (mit `boden = 0.05`). Tragen Sie in **Tabelle C** das SNR nachher ein und beurteilen Sie **Verständlichkeit** und **Klang** (Artefakte, musikalisches Rauschen, dumpf). Bei welcher Einstellung war das SNR am größten? War das auch die Einstellung, die Ihnen am besten gefallen hat?

#### Zusatzaufgabe Z3.2: Boden und Rauschstärke (optional)

> **Z3.2:** Ändern Sie `boden` auf 0.01 und 0.2. Setzen Sie danach in 4.3 `ziel_snr_db` auf 0 und auf 20 und führen Sie die Zellen in 4.3 und 4.4 erneut aus. Wie verändern sich SNR und Klang?

### 4.5 Equalizer: lauter oder besser? (Aufgabe 3.4, Pflicht)

**a) Lauter ist nicht besser.** Wird die verrauschte Aufnahme mit `verstaerkung` multipliziert, werden **Sprache und Rauschen gleich stark** lauter. Ihr Verhältnis und damit das SNR bleiben gleich. Werte über 1 kann ein Wandler nicht darstellen. Wir schneiden sie hier **absichtlich** mit `np.clip` ab. Diese künstlich erzeugte Übersteuerung verzerrt das Signal.

In [ ]:
verstaerkung = 4        # Vorschläge: 2, 4 (gültig: größer als 0)
if verstaerkung <= 0:
    raise ValueError("verstaerkung muss größer als 0 sein.")

lauter = verstaerkung * verrauscht
print("Spitzenwert nach Verstärkung:", round(np.max(np.abs(lauter)), 2))
print("SNR, Sprache und Rauschen beide verstärkt:", round(snr_db(verstaerkung * sprache, lauter), 1), "dB")

uebersteuert = np.clip(lauter, -1, 1)          # absichtlich erzeugte Übersteuerung
print("Durch np.clip abgeschnittene Samples:", np.sum(np.abs(lauter) > 1))
print("SNR nach der Übersteuerung:", round(snr_db(verstaerkung * sprache, uebersteuert), 1), "dB")

# Hörvergleich bei gleichem Pegel: die übersteuerte Version wird wieder durch die Verstärkung geteilt
anhoeren(verrauscht, sr_sprache, "Verrauscht (ursprünglicher Pegel):")
anhoeren(uebersteuert / verstaerkung, sr_sprache, "Verstärkt, übersteuert und wieder leiser gemacht:")

**b) Equalizer.** Ein **Equalizer** (wie am Mischpult) verstärkt oder dämpft Frequenzbereiche. Unser 3-Band-Equalizer teilt das Signal auf:
1. **Tiefen**: unter 250 Hz (Tiefpass)
2. **Höhen**: über 4000 Hz (Hochpass)
3. **Mitten**: der Rest (Original minus Tiefen minus Höhen)

Jedes Band wird mit einer Verstärkung in dB multipliziert. +6 dB ist ungefähr die **doppelte** Amplitude, -6 dB ungefähr die **halbe**, 0 dB lässt das Band unverändert. Gültig sind hier -24 dB bis +12 dB. Der Equalizer verändert den **Klang**. Er ist kein Verfahren zur Rauschunterdrückung.

Wir hören auf zwei Arten:
- **Gleiche Skala:** Pegelunterschiede bleiben hörbar. Eine Anhebung macht die Musik oft auch lauter.
- **Gleicher RMS-Wert:** Beide Versionen haben denselben Signalpegel. So vergleicht man eher die **Klangfarbe**. Gleicher RMS bedeutet aber **nicht sicher gleiche empfundene Lautstärke**, weil das Gehör Frequenzen unterschiedlich stark wahrnimmt.

In [ ]:
def equalizer(x, sr, tiefen_db=0, mitten_db=0, hoehen_db=0):
    """3-Band-Equalizer: Signal aufteilen, jedes Band verstärken, wieder addieren."""
    for wert in [tiefen_db, mitten_db, hoehen_db]:
        if not -24 <= wert <= 12:
            raise ValueError("Verstärkungen müssen zwischen -24 dB und +12 dB liegen.")
    tiefen = tiefpass(x, sr, 250)
    hoehen = hochpass(x, sr, 4000)
    mitten = x - tiefen - hoehen                   # der Rest ist das mittlere Band
    # dB in einen Faktor umrechnen: Faktor = 10 hoch (dB / 20)
    return (10**(tiefen_db / 20) * tiefen
            + 10**(mitten_db / 20) * mitten
            + 10**(hoehen_db / 20) * hoehen)

musik, sr_musik = sf.read("data/music.wav")
musik_kurz = musik[:sekunden_zu_samples(8.0, sr_musik)]     # erste 8 Sekunden, beide Kanäle

In [ ]:
tiefen_db = 6      # Tiefen in dB (Tabelle D)
mitten_db = 0      # Mitten in dB
hoehen_db = 0      # Höhen in dB

musik_eq = equalizer(musik_kurz, sr_musik, tiefen_db, mitten_db, hoehen_db)

skala_eq = gemeinsame_skala(musik_kurz, musik_eq)
print("Gemeinsame Skala:", round(skala_eq, 3))
anhoeren(musik_kurz, sr_musik, "Original (gleiche Skala):", skala_eq)
anhoeren(musik_eq, sr_musik, "Equalizer (gleiche Skala):", skala_eq)

rms_original = np.sqrt(np.mean(musik_kurz**2))
rms_eq = np.sqrt(np.mean(musik_eq**2))
musik_eq_gleicher_rms = musik_eq * (rms_original / rms_eq)
print("RMS Original:", round(rms_original, 4), "  RMS Equalizer:", round(rms_eq, 4))

skala_rms = gemeinsame_skala(musik_kurz, musik_eq_gleicher_rms)
anhoeren(musik_kurz, sr_musik, "Original (Vergleich mit gleichem RMS):", skala_rms)
anhoeren(musik_eq_gleicher_rms, sr_musik, "Equalizer mit gleichem RMS wie das Original:", skala_rms)

> **Aufgabe 3.4:**
> a) Probieren Sie `verstaerkung` = 2 und 4. Wie viele Samples schneidet `np.clip` jeweils ab? Wie klingt die Übersteuerung? Hat sich das SNR durch die Verstärkung allein verändert?
> b) Stellen Sie die Equalizer-Einstellungen aus **Tabelle D** ein. Beschreiben Sie den Klang einmal mit gleicher Skala und einmal mit gleichem RMS-Wert. Welche Version fanden Sie jeweils "besser", und lag das eher an der Lautstärke oder am Klang?

#### Zusatzaufgabe Z3.3: Ergebnis als WAV-Datei speichern (optional)

Die Datei landet im Ordner `ausgabe/`. In Google Colab finden Sie sie links unter **Dateien** (Ordnersymbol) und können sie mit Rechtsklick herunterladen. Nach dem Löschen der Laufzeit ist sie weg.

In [ ]:
import os
os.makedirs("ausgabe", exist_ok=True)

skala_datei = gemeinsame_skala(entrauscht)     # verhindert Übersteuerung in der Datei
sf.write("ausgabe/sprache_entrauscht.wav", entrauscht * skala_datei, sr_sprache, subtype="PCM_16")
print("Gespeichert: ausgabe/sprache_entrauscht.wav  (Skala:", round(skala_datei, 3), ")")

## 5 Auswertung und Diskussion

**Tabelle A: Brummen und Hochpass (Aufgabe 3.1)**

| grenzfrequenz_brumm [Hz] | SNR vorher [dB] | SNR nachher [dB] | gestiegen / gesunken | Brummen hörbar? | Stimme verändert? |
|---|---|---|---|---|---|
| 60 | | | | | |
| 120 | | | | | |
| 200 | | | | | |
| 400 | | | | | |
| 800 | | | | | |

**Tabelle B: Tiefpass bei breitbandigem Rauschen (Aufgabe 3.2)**

| grenzfrequenz_tp [Hz] | SNR vorher [dB] | SNR nachher [dB] | Rauschen hörbar? | Sprache verändert? |
|---|---|---|---|---|
| 1000 | | | | |
| 3000 | | | | |
| 6000 | | | | |

**Tabelle C: Spektrale Subtraktion, `ziel_snr_db = 10`, `boden = 0.05` (Aufgabe 3.3)**

| staerke | SNR vorher [dB] | SNR nachher [dB] | Verständlichkeit | Klang, Artefakte |
|---|---|---|---|---|
| 0.5 | | | | |
| 1.0 | | | | |
| 2.0 | | | | |
| 4.0 | | | | |

**Tabelle D: Equalizer (Aufgabe 3.4 b)**

| tiefen_db | mitten_db | hoehen_db | Klang bei gleicher Skala | Klang bei gleichem RMS |
|---|---|---|---|---|
| +6 | 0 | 0 | | |
| -12 | 0 | 0 | | |
| 0 | 0 | +6 | | |
| 0 | 0 | -12 | | |

**Fragen**
1. Warum kann ein Hochpass ein Brummen bei 50 Hz verringern, aber kein weißes Rauschen im Sprachbereich?
2. Wie hat sich das SNR in Aufgabe 3.1 mit steigender Grenzfrequenz verändert? Erklären Sie Ihre Beobachtung.
3. Hat die Einstellung mit dem größten SNR in Aufgabe 3.3 auch am besten geklungen? Erklären Sie mit den Begriffen Signalähnlichkeit, Verständlichkeit und empfundene Qualität.
4. Beschreiben Sie das musikalische Rauschen, falls Sie es gehört haben. Bei welcher Einstellung war es am deutlichsten?
5. Was hat die Verstärkung in Aufgabe 3.4 a) am SNR verändert, und was hat die Übersteuerung verändert?
6. Warum ist ein Vergleich mit gleichem RMS-Wert fairer als ein Vergleich mit gleicher Skala, und warum ist er trotzdem nicht perfekt?

**Ihre Antworten:**
1. 
2. 
3. 
4. 
5. 
6. 

---
## 6 Quellen und Audiodaten

Die Aufnahmen stammen von Dritten. Für dieses Labor wurden nur Ausschnitte herausgeschnitten und als WAV-Dateien gespeichert (20 ms Ein- und Ausblenden an den Schnittstellen, sonst keine Bearbeitung). Alle Details stehen in [`DATA_SOURCES.md`](https://github.com/Lamhour-Mohamed-Akram/audio-dsp-lab/blob/main/DATA_SOURCES.md).

- **`speech.wav`** (Sprache, 16000 Hz, mono, 15,9 s): LibriVox-Hörbuch "Fabeln" von Magnus Gottfried Lichtwer, Abschnitt "Das Wiesel und die Hühner", gelesen von **marham63**. Lizenz: **Public Domain**. Quelle: <https://archive.org/details/fabeln_1905_librivox>. Ausschnitt 12,8 s bis 28,7 s, umgetastet auf 16000 Hz.
- **`music.wav`** (Musik, 44100 Hz, stereo, 20,0 s): Julius Fučík, "Entrance of the Gladiators" ("Einzug der Gladiatoren"), gespielt von der **United States Marine Band**. Lizenz: **Public Domain** (Werk der US-Regierung). Quelle: <https://commons.wikimedia.org/wiki/File:Julius_Fu%C4%8D%C3%ADk%27s_%22Entrance_of_the_Gladiators%22,_performed_by_the_United_States_Marine_Band.flac>. Ausschnitt 24 s bis 44 s, als 16-Bit-WAV gespeichert.